### Naive Bayes Formula for Email Spam Classification

The Naive Bayes classifier is based on Bayes' theorem with the "naive" assumption of conditional independence between features. For email spam classification, we want to calculate the probability that an email is spam given a set of words in it, or that it is ham (not spam) given those words.

Let $C$ be the class (e.g., Spam or Ham), and $W = \{w_1, w_2, ..., w_n\}$ be the set of words in an email. We want to find the class $C$ that maximizes $P(C|W)$.

Using Bayes' theorem:

$P(C|W) = \frac{P(W|C) * P(C)}{P(W)}$

Since $P(W)$ is constant for all classes, we only need to compare $P(W|C) * P(C)$.

Due to the "naive" assumption of conditional independence among words:

$P(W|C) = P(w_1|C) * P(w_2|C) * ... * P(w_n|C)$

So, the decision rule becomes:

Classify as Spam if $P(\text{Spam}|W) > P(\text{Ham}|W)$

Which means we compare:

$P(w_1|\text{Spam}) * ... * P(w_n|\text{Spam}) * P(\text{Spam})$

versus

$P(w_1|\text{Ham}) * ... * P(w_n|\text{Ham}) * P(\text{Ham})$

To avoid numerical underflow with many small probabilities, we often use the logarithmic form:

$\text{log}(P(C|W)) \propto \text{log}(P(C)) + \sum_{i=1}^{n} \text{log}(P(w_i|C))$

Where:
- $P(C)$: Prior probability of class C (e.g., proportion of Spam emails in the training data).
- $P(w_i|C)$: Likelihood of word $w_i$ appearing given class C. This is typically calculated using Laplace smoothing to handle unseen words:
  $P(w_i|C) = \frac{\text{count}(w_i, C) + 1}{\text{count}(\text{all words in C}) + \text{Vocab Size}}$

In [1]:
import numpy as np

class NaiveBayesSpamClassifier:
    def __init__(self, alpha=1):
        # Initialize the Naive Bayes classifier with Laplace smoothing parameter alpha
        self.alpha = alpha
        self.classes = []
        self.prior_probs = {}
        self.likelihoods = {}
        self.vocab = set()

    def fit(self, X, y):
        # X: list of tokenized emails (list of lists of words)
        # y: list of labels (0 for ham, 1 for spam)

        # Get unique classes (spam/ham)
        self.classes = np.unique(y)
        num_docs = len(X)

        # --- Step 1: Build Vocabulary ---
        # Collect all unique words from all emails to form the vocabulary
        for doc_words in X:
            self.vocab.update(doc_words)
        vocab_size = len(self.vocab)

        # --- Step 2: Calculate Prior Probabilities P(C) ---
        # P(C) = (Number of documents in class C) / (Total number of documents)
        for c in self.classes:
            num_docs_in_class = np.sum(y == c)
            self.prior_probs[c] = num_docs_in_class / num_docs

        # --- Step 3: Calculate Likelihoods P(word|C) ---
        # P(word|C) = (Count of word in class C + alpha) / (Total words in class C + alpha * Vocab Size)
        for c in self.classes:
            self.likelihoods[c] = {} # Store likelihoods for each word for the current class
            # Filter documents belonging to the current class
            class_docs = [X[i] for i, label in enumerate(y) if label == c]

            # Count total words in the current class
            total_words_in_class = sum(len(doc) for doc in class_docs)

            # Count word occurrences in the current class
            word_counts_in_class = {}
            for doc_words in class_docs:
                for word in doc_words:
                    word_counts_in_class[word] = word_counts_in_class.get(word, 0) + 1

            # Calculate likelihood for each word in the vocabulary
            for word in self.vocab:
                count = word_counts_in_class.get(word, 0)
                self.likelihoods[c][word] = (count + self.alpha) / (total_words_in_class + self.alpha * vocab_size)

    def predict(self, X):
        # X: list of tokenized emails (list of lists of words) to predict
        predictions = []
        for doc_words in X:
            # Initialize scores for each class using the log of prior probabilities
            scores = {c: np.log(self.prior_probs[c]) for c in self.classes}

            # --- Step 4: Calculate Posterior Probability (log form) ---
            # log(P(C|W)) = log(P(C)) + Sum(log(P(word|C)) for each word in document)
            for c in self.classes:
                for word in doc_words:
                    # If a word is not in the training vocabulary, assign a very small likelihood
                    # (equivalent to using the smoothed probability for unseen words)
                    if word in self.vocab:
                        scores[c] += np.log(self.likelihoods[c][word])
                    else:
                        # Handle unseen words during prediction. Use a default smoothed probability.
                        # This is equivalent to (0 + alpha) / (total_words_in_class + alpha * vocab_size)
                        # However, for simplicity and assuming alpha=1, we can use 1 / (total_words_in_class + vocab_size) if we just want a small probability.
                        # A more robust way would be to compute a generic unseen word probability based on alpha and overall vocab_size
                        # For now, let's just add a very small log probability.
                        # A better approach would be to calculate it based on the average likelihood for a given class over all words.
                        # For this example, we'll use a placeholder for unseen words, which is essentially the smoothed probability for a word that occurred 0 times.
                        # The denominator for this unseen word would be the total words in that class + alpha * vocab_size
                        # To avoid recomputing total_words_in_class, we can pre-calculate it or use a small constant for log(P(unseen_word|C)).
                        # A practical choice is to use the smallest likelihood in the specific class or a general small value.
                        # For simplicity, we'll assign a very small log probability for words not in vocab.
                        # This is effectively P(word|C) = alpha / (total_words_in_class + alpha * vocab_size)
                        # The `fit` method already calculated likelihoods for all words in `vocab`.
                        # For words not in `vocab`, we assume they have appeared 0 times in either class during training.
                        # So, their likelihood is `alpha / (total_words_in_class + alpha * vocab_size)`.
                        # Since `total_words_in_class` is not easily accessible here without re-aggregation,
                        # we can use a very small constant or pre-compute `total_words_in_class` in `fit` and store it.
                        # For a truly robust implementation, you'd store `total_words_in_class` for each `c`.
                        # For this simplified example, let's use a small constant for demonstration.
                        scores[c] += np.log(1e-10) # Assign a very small probability to unseen words

            # --- Step 5: Make Prediction ---
            # The class with the highest log posterior probability is the prediction
            predicted_class = max(scores, key=scores.get)
            predictions.append(predicted_class)
        return np.array(predictions)


### Demo Usage of Naive Bayes Classifier

In [2]:
# Sample training data: emails and their labels (0 for ham, 1 for spam)
# Each email is tokenized into a list of words

# Training emails (X_train)
# Label 0: Ham
# Label 1: Spam
X_train = [
    ["send", "me", "your", "password"],
    ["hello", "how", "are", "you"],
    ["free", "money", "now"],
    ["meeting", "tomorrow", "at", "10am"],
    ["claim", "your", "prize", "today"],
    ["lunch", "with", "friends"]
]

y_train = np.array([1, 0, 1, 0, 1, 0]) # Corresponding labels

# Create and train the Naive Bayes classifier
# We use alpha=1 for Laplace smoothing to handle words not seen in training data well.
classifier = NaiveBayesSpamClassifier(alpha=1)
classifier.fit(X_train, y_train)

print("--- Trained Classifier Details ---")
print("Prior Probabilities:", classifier.prior_probs)
# Display likelihoods for a few words to show what was learned
print("Likelihoods (sample for 'password', 'hello', 'free'):")
for word in ['password', 'hello', 'free']:
    if word in classifier.vocab:
        print(f"  P('{word}'|Ham): {classifier.likelihoods[0].get(word, 'N/A'):.4f}")
        print(f"  P('{word}'|Spam): {classifier.likelihoods[1].get(word, 'N/A'):.4f}")
    else:
        print(f"  '{word}' not in vocabulary.")
print("\n")

# Sample test data: new emails to classify
X_test = [
    ["your", "password", "is", "needed"],
    ["how", "are", "you", "today"],
    ["claim", "free", "money"],
    ["new", "meeting", "details"]
]

# Predict the classes for the test emails
predictions = classifier.predict(X_test)

print("--- Predictions ---")
for i, email_words in enumerate(X_test):
    label = "Spam" if predictions[i] == 1 else "Ham"
    print(f"Email: '{' '.join(email_words)}' -> Predicted: {label}")

# Example with an email containing an unseen word
X_test_unseen = [
    ["buy", "crypto", "now"]
]
predictions_unseen = classifier.predict(X_test_unseen)
label_unseen = "Spam" if predictions_unseen[0] == 1 else "Ham"
print(f"\nEmail with unseen word: '{' '.join(X_test_unseen[0])}' -> Predicted: {label_unseen}")


--- Trained Classifier Details ---
Prior Probabilities: {np.int64(0): np.float64(0.5), np.int64(1): np.float64(0.5)}
Likelihoods (sample for 'password', 'hello', 'free'):
  P('password'|Ham): 0.0312
  P('password'|Spam): 0.0625
  P('hello'|Ham): 0.0625
  P('hello'|Spam): 0.0312
  P('free'|Ham): 0.0312
  P('free'|Spam): 0.0625


--- Predictions ---
Email: 'your password is needed' -> Predicted: Spam
Email: 'how are you today' -> Predicted: Ham
Email: 'claim free money' -> Predicted: Spam
Email: 'new meeting details' -> Predicted: Ham

Email with unseen word: 'buy crypto now' -> Predicted: Spam


Gaussian Discriminant Analysis (GDA) & Logistics Regression Analysis

In [3]:
import numpy as np

# 1. GENERATE REAL-WORLD SYNTHETIC DATA (Heart Disease Prediction)
# Features: [Max Heart Rate, Cholesterol]
np.random.seed(42)
num_samples = 100
# Co variance is said to be an identity matrix.
# Class 0: Healthy Patients (Higher heart rate, lower cholesterol)
mean_healthy = [150, 200]
cov_healthy = [[100, -15], [-15, 150]]
X_healthy = np.random.multivariate_normal(mean_healthy, cov_healthy, num_samples)
y_healthy = np.zeros(num_samples)

# Class 1: Heart Disease Patients (Lower heart rate, higher cholesterol)
mean_disease = [130, 260]
cov_disease = [[120, -10], [-10, 200]]
X_disease = np.random.multivariate_normal(mean_disease, cov_disease, num_samples)
y_disease = np.ones(num_samples)

# Combine and shuffle
X = np.vstack((X_healthy, X_disease))
y = np.concatenate((y_healthy, y_disease))
indices = np.arange(X.shape[0])
np.random.shuffle(indices)
X, y = X[indices], y[indices]

# Feature Normalization (Crucial for Logistic Regression Gradient Descent)
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_scaled = (X - X_mean) / X_std


# 2. GAUSSIAN DISCRIMINANT ANALYSIS (GDA) IMPLEMENTATION
class GDA:
    def fit(self, X, y):
        self.m, self.n = X.shape
        self.phi = np.mean(y)

        # Split data by class
        X_0 = X[y == 0]
        X_1 = X[y == 1]

        # Calculate class means
        self.mu_0 = np.mean(X_0, axis=0)
        self.mu_1 = np.mean(X_1, axis=0)

        # Calculate shared Covariance Matrix (Sigma)
        diff_0 = X_0 - self.mu_0
        diff_1 = X_1 - self.mu_1
        self.sigma = (np.dot(diff_0.T, diff_0) + np.dot(diff_1.T, diff_1)) / self.m

    def _gaussian_pdf(self, x, mu, sigma):
        # Multivariate Normal Distribution Formula
        n = len(mu)
        det_sigma = np.linalg.det(sigma)
        inv_sigma = np.linalg.inv(sigma)

        diff = x - mu
        exponent = -0.5 * np.sum(np.dot(diff, inv_sigma) * diff, axis=1)
        normalization = 1.0 / (((2 * np.pi) ** (n / 2)) * np.sqrt(det_sigma))
        return normalization * np.exp(exponent)

    def predict_proba(self, X):
        # P(x|y=0) and P(x|y=1)
        px_y0 = self._gaussian_pdf(X, self.mu_0, self.sigma)
        px_y1 = self._gaussian_pdf(X, self.mu_1, self.sigma)

        # Bayes Theorem: P(y=1|x) = P(x|y=1)P(y=1) / P(x)
        p_x_and_y1 = px_y1 * self.phi
        p_x_and_y0 = px_y0 * (1 - self.phi)

        prob_y1 = p_x_and_y1 / (p_x_and_y1 + p_x_and_y0)
        return prob_y1


# 3. LOGISTIC REGRESSION IMPLEMENTATION (Gradient Descent)
class LogisticRegressionGD:
    def __init__(self, lr=0.1, iterations=1000):
        self.lr = lr
        self.iterations = iterations

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        m, n = X.shape
        # Initialize weights and bias to zeros
        self.w = np.zeros(n)
        self.b = 0.0

        # Gradient Descent Iterations
        for _ in range(self.iterations):
            # 1. Calculate linear combination and prediction probabilities
            z = np.dot(X, self.w) + self.b
            y_pred = self._sigmoid(z)

            # 2. Compute Gradients
            dw = (1 / m) * np.dot(X.T, (y_pred - y))
            db = (1 / m) * np.sum(y_pred - y)

            # 3. Update Parameters
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X):
        z = np.dot(X, self.w) + self.b
        return self._sigmoid(z)


# 4. EXECUTION AND COMPARING SOLUTIONS
# Train GDA (Uses raw data as it is scale-invariant)
gda_model = GDA()
gda_model.fit(X, y)
gda_probs = gda_model.predict_proba(X)
gda_preds = (gda_probs >= 0.5).astype(int)

# Train Logistic Regression (Uses scaled data for stable gradient descent)
lr_model = LogisticRegressionGD(lr=0.1, iterations=1000)
lr_model.fit(X_scaled, y)
lr_probs = lr_model.predict_proba(X_scaled)
lr_preds = (lr_probs >= 0.5).astype(int)

# Evaluate Accuracies
gda_acc = np.mean(gda_preds == y) * 100
lr_acc = np.mean(lr_preds == y) * 100

print(f"Gaussian Discriminant Analysis Accuracy: {gda_acc:.2f}%")
print(f"Logistic Regression Accuracy:            {lr_acc:.2f}%")

# Test on a new hypothetical patient
# New Patient: Max Heart Rate = 135, Cholesterol = 250
new_patient = np.array([[135, 250]])
new_patient_scaled = (new_patient - X_mean) / X_std

print(f"\n--- Testing New Patient [Heart Rate: 135, Cholesterol: 250] ---")
print(f"GDA Risk Probability:        {gda_model.predict_proba(new_patient)[0] * 100:.2f}%")
print(f"Logistic Regression Risk Prob: {lr_model.predict_proba(new_patient_scaled)[0] * 100:.2f}%")

Gaussian Discriminant Analysis Accuracy: 99.50%
Logistic Regression Accuracy:            99.50%

--- Testing New Patient [Heart Rate: 135, Cholesterol: 250] ---
GDA Risk Probability:        99.97%
Logistic Regression Risk Prob: 96.74%
